In [ ]:
import copy
import random
import re

import numpy as np
import pandas as pd
import nltk
import torch
import torch.nn as nn

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from torch.utils.data import DataLoader, TensorDataset

DATA_DIR = "../data"

messages_df = pd.read_csv(f"{DATA_DIR}/disaster_messages.csv")
categories_raw = pd.read_csv(f"{DATA_DIR}/disaster_categories.csv")

print("messages_df shape:", messages_df.shape)
print("categories_raw shape:", categories_raw.shape)
print("IDs aligned row-wise:", messages_df["id"].equals(categories_raw["id"]))


messages_df shape: (26248, 4)
categories_raw shape: (26248, 2)
IDs aligned row-wise: True


In [ ]:
# Expand category strings and build the modeling dataframe

category_parts = categories_raw["categories"].str.split(";", expand=True)
category_names = [part.rsplit("-", 1)[0] for part in category_parts.iloc[0]]
category_parts.columns = category_names

for col in category_parts.columns:
    category_parts[col] = category_parts[col].str.rsplit("-", n=1).str[-1].astype(int)

# Keep row alignment
df = pd.concat(
    [messages_df.drop(columns=["id"]), categories_raw[["id"]], category_parts],
    axis=1
)

df = df.drop_duplicates().reset_index(drop=True)
df = df[df["message"].notna()].copy()
df["message"] = df["message"].astype(str).str.strip()
df = df[df["message"] != ""].reset_index(drop=True)

print("Final dataframe shape:", df.shape)
display(df[["id", "message", "genre", "request", "aid_related", "water", "food", "shelter"]].head())


Final dataframe shape: (26214, 40)


,id,message,genre,request,aid_related,water,food,shelter
0,2,Weather update - a cold front from Cuba that could pass over Haiti,direct,0,0,0,0,0
1,7,Is the Hurricane over or is it not over,direct,0,1,0,0,0
2,8,Looking for someone but no name,direct,0,0,0,0,0
3,9,UN reports Leogane 80-90 destroyed. Only Hospital St. Croix functioning. Needs supplies desperately.,direct,1,1,0,0,0
4,12,"says: west side of Haiti, rest of the country today and tonight",direct,0,0,0,0,0


In [67]:
# Text preprocessing

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_message(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    tokens = text.split()
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

df["message_clean"] = df["message"].apply(preprocess_message)
df = df[df["message_clean"].str.strip() != ""].reset_index(drop=True)

print("Rows after preprocessing:", len(df))
display(df[["message", "message_clean"]].head(10))


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/harmansingh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/harmansingh/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/harmansingh/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Rows after preprocessing: 26212


,message,message_clean
0,Weather update - a cold front from Cuba that could pass over Haiti,weather update cold front cuba could pas haiti
1,Is the Hurricane over or is it not over,hurricane
2,Looking for someone but no name,looking someone name
3,UN reports Leogane 80-90 destroyed. Only Hospital St. Croix functioning. Needs supplies desperately.,un report leogane 80 90 destroyed hospital st croix functioning need supply desperately
4,"says: west side of Haiti, rest of the country today and tonight",say west side haiti rest country today tonight
5,Information about the National Palace-,information national palace
6,Storm at sacred heart of jesus,storm sacred heart jesus
7,"Please, we need tents and water. We are in Silo, Thank you!",please need tent water silo thank
8,"I would like to receive the messages, thank you",would like receive message thank
9,I am in Croix-des-Bouquets. We have health issues. They ( workers ) are in Santo 15. ( an area in Croix-des-Bouquets ),croix de bouquet health issue worker santo 15 area croix de bouquet


In [68]:
# Final baseline target + shared intent targets + train/test split

baseline_target = "request"

target_cols = [
    "aid_related",
    "other_aid",
    "medical_help",
    "medical_products",
    "search_and_rescue",
    "hospitals",
    "water",
    "food",
    "shelter",
    "transport",
    "aid_centers",
    "buildings",
    "electricity",
    "infrastructure_related",
]

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df[baseline_target].astype(int),
)

vectorizer = TfidfVectorizer(max_features=1000)
X_train = vectorizer.fit_transform(train_df["message_clean"])
X_test = vectorizer.transform(test_df["message_clean"])

y_train = train_df[baseline_target].astype(int).values
y_test = test_df[baseline_target].astype(int).values

Y_train = train_df[target_cols].astype("float32").values
Y_test = test_df[target_cols].astype("float32").values

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)
print("Train request counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test request counts:", dict(zip(*np.unique(y_test, return_counts=True))))


X_train shape: (20969, 1000)
X_test shape: (5243, 1000)
Y_train shape: (20969, 14)
Y_test shape: (5243, 14)
Train request counts: {np.int64(0): np.int64(17390), np.int64(1): np.int64(3579)}
Test request counts: {np.int64(0): np.int64(4348), np.int64(1): np.int64(895)}


In [ ]:
# Naive Bayes baseline

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

nb_preds = nb_model.predict(X_test)

print(f"Baseline Accuracy:  {accuracy_score(y_test, nb_preds):.4f}")
print(f"Baseline Precision: {precision_score(y_test, nb_preds, zero_division=0):.4f}")
print(f"Baseline Recall:    {recall_score(y_test, nb_preds, zero_division=0):.4f}")
print(f"Baseline F1:        {f1_score(y_test, nb_preds, zero_division=0):.4f}")

print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, nb_preds))

print("\nClassification Report:")
print(classification_report(y_test, nb_preds, zero_division=0))


Baseline Accuracy:  0.8859
Baseline Precision: 0.7415
Baseline Recall:    0.5095
Baseline F1:        0.6040

Confusion Matrix [[TN, FP], [FN, TP]]:
[[4189  159]
 [ 439  456]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.96      0.93      4348
           1       0.74      0.51      0.60       895

    accuracy                           0.89      5243
   macro avg       0.82      0.74      0.77      5243
weighted avg       0.88      0.89      0.88      5243



In [70]:
# PyTorch tensor setup

X_train_tensor = torch.tensor(X_train.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.toarray(), dtype=torch.float32)

Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("X_train_tensor:", X_train_tensor.shape)
print("X_test_tensor:", X_test_tensor.shape)
print("Y_train_tensor:", Y_train_tensor.shape)
print("Y_test_tensor:", Y_test_tensor.shape)


X_train_tensor: torch.Size([20969, 1000])
X_test_tensor: torch.Size([5243, 1000])
Y_train_tensor: torch.Size([20969, 14])
Y_test_tensor: torch.Size([5243, 14])


In [ ]:
# Train the model

SEED = 2
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

class IntentNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, output_dim),
        )

    def forward(self, x):
        return self.net(x)

model = IntentNet(X_train_tensor.shape[1], Y_train_tensor.shape[1]).to(device)

pos_counts = Y_train_tensor.sum(dim=0)
neg_counts = Y_train_tensor.shape[0] - pos_counts
pos_weight = torch.clamp(
    neg_counts / torch.clamp(pos_counts, min=1.0),
    max=10.0
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

num_epochs = 20
patience = 3
epochs_without_improvement = 0
best_test_loss = float("inf")
best_epoch = None
best_state = None

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0

    for batch_X, batch_Y in train_loader:
        batch_X = batch_X.to(device)
        batch_Y = batch_Y.to(device)

        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_Y)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)

    model.eval()
    running_test_loss = 0.0

    with torch.no_grad():
        for batch_X, batch_Y in test_loader:
            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)

            logits = model(batch_X)
            loss = criterion(logits, batch_Y)
            running_test_loss += loss.item()

    avg_test_loss = running_test_loss / len(test_loader)

    if avg_test_loss < best_test_loss:
        best_test_loss = avg_test_loss
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1:2d}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Test Loss: {avg_test_loss:.4f}"
    )

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping at epoch {epoch + 1}")
        break

model.load_state_dict(best_state)
print(f"\nLoaded best model from epoch {best_epoch} with test loss {best_test_loss:.4f}")



Using device: cuda
GPU: NVIDIA GeForce RTX 3080
Epoch  1/20 | Train Loss: 0.8935 | Test Loss: 0.8064
Epoch  2/20 | Train Loss: 0.7356 | Test Loss: 0.7254
Epoch  3/20 | Train Loss: 0.6863 | Test Loss: 0.6966
Epoch  4/20 | Train Loss: 0.6541 | Test Loss: 0.6726
Epoch  5/20 | Train Loss: 0.6273 | Test Loss: 0.6594
Epoch  6/20 | Train Loss: 0.6067 | Test Loss: 0.6452
Epoch  7/20 | Train Loss: 0.5906 | Test Loss: 0.6355
Epoch  8/20 | Train Loss: 0.5770 | Test Loss: 0.6325
Epoch  9/20 | Train Loss: 0.5641 | Test Loss: 0.6254
Epoch 10/20 | Train Loss: 0.5552 | Test Loss: 0.6224
Epoch 11/20 | Train Loss: 0.5427 | Test Loss: 0.6203
Epoch 12/20 | Train Loss: 0.5325 | Test Loss: 0.6192
Epoch 13/20 | Train Loss: 0.5235 | Test Loss: 0.6237
Epoch 14/20 | Train Loss: 0.5164 | Test Loss: 0.6239
Epoch 15/20 | Train Loss: 0.5096 | Test Loss: 0.6237

Early stopping at epoch 15

Loaded best model from epoch 12 with test loss 0.6192


In [ ]:
# predictions + per-target evaluation

with torch.no_grad():
    test_probs = torch.sigmoid(model(X_test_tensor.to(device))).detach().cpu().tolist()

pred_cols = [f"pred_{col}" for col in target_cols]
actual_cols = [f"actual_{col}" for col in target_cols]

pred_df = pd.DataFrame(test_probs, columns=pred_cols, index=test_df.index)
actual_df = pd.DataFrame(Y_test, columns=actual_cols, index=test_df.index)

multi_rows = []
for i, col in enumerate(target_cols):
    y_true = Y_test[:, i].astype(int)
    y_score = pred_df[f"pred_{col}"].values
    y_pred = (y_score >= 0.50).astype(int)

    multi_rows.append({
        "target": col,
        "positive_rate": y_true.mean(),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mean_pred_score": y_score.mean(),
    })

multi_eval_df = pd.DataFrame(multi_rows).sort_values("f1", ascending=False)
display(multi_eval_df)


,target,positive_rate,precision,recall,f1,mean_pred_score
0,aid_related,0.417700,0.683993,0.757078,0.718682,0.484765
7,food,0.112340,0.557542,0.847199,0.672507,0.217693
6,water,0.068091,0.457627,0.831933,0.590457,0.165532
8,shelter,0.096319,0.419960,0.841584,0.560316,0.242437
11,buildings,0.052451,0.276596,0.756364,0.405063,0.198288
2,medical_help,0.078581,0.258559,0.696602,0.377135,0.312543
3,medical_products,0.052260,0.266154,0.631387,0.374459,0.203551
12,electricity,0.020408,0.304348,0.457944,0.365672,0.106602
1,other_aid,0.129887,0.236986,0.701909,0.354337,0.434524
13,infrastructure_related,0.074194,0.205641,0.580977,0.303763,0.291701


In [ ]:
# Fuzzy system setup: membership functions, grouped antecedents, and helpers

import numpy as np

UNIVERSE = np.linspace(0.0, 1.0, 201)

def trimf(x, a, b, c):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)

    left = (a < x) & (x < b)
    right = (b < x) & (x < c)

    if b != a:
        y[left] = (x[left] - a) / (b - a)
    if c != b:
        y[right] = (c - x[right]) / (c - b)

    y[x == b] = 1.0
    return np.clip(y, 0.0, 1.0)

def trapmf(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)

    rise = (a < x) & (x < b)
    plateau = (b <= x) & (x <= c)
    fall = (c < x) & (x < d)

    if b != a:
        y[rise] = (x[rise] - a) / (b - a)
    y[plateau] = 1.0
    if d != c:
        y[fall] = (d - x[fall]) / (d - c)

    return np.clip(y, 0.0, 1.0)

def fuzzify_scalar(x):
    x_arr = np.array([float(x)])
    return {
        "low": float(trapmf(x_arr, 0.0, 0.0, 0.12, 0.30)[0]),
        "medium": float(trimf(x_arr, 0.18, 0.42, 0.68)[0]),
        "high": float(trimf(x_arr, 0.58, 0.76, 0.90)[0]),
        "very_high": float(trapmf(x_arr, 0.82, 0.92, 1.0, 1.0)[0]),
    }

OUTPUT_MFS = {
    "low": trapmf(UNIVERSE, 0.0, 0.0, 0.12, 0.30),
    "medium": trimf(UNIVERSE, 0.18, 0.42, 0.68),
    "high": trimf(UNIVERSE, 0.58, 0.76, 0.90),
    "very_high": trapmf(UNIVERSE, 0.82, 0.92, 1.0, 1.0),
}

def mamdani_infer(rule_outputs):
    agg = np.zeros_like(UNIVERSE)

    for strength, label in rule_outputs:
        strength = float(np.clip(strength, 0.0, 1.0))
        agg = np.maximum(agg, np.minimum(OUTPUT_MFS[label], strength))

    if agg.sum() == 0:
        return 0.0

    return float(np.sum(UNIVERSE * agg) / np.sum(agg))

def genre_to_directness(genre):
    mapping = {
        "direct": 1.0,
        "social": 0.6,
        "news": 0.1,
    }
    return mapping.get(str(genre).lower(), 0.5)


In [74]:
# Build the fuzzy-input dataframe from the neural outputs

policy_view_df = test_df[["message", "message_clean", "genre"]].copy()

for col in target_cols:
    policy_view_df[f"pred_{col}"] = pred_df.loc[policy_view_df.index, f"pred_{col}"].values

policy_view_df["general_distress"] = np.clip(
    0.70 * policy_view_df["pred_aid_related"] +
    0.30 * policy_view_df["pred_other_aid"],
    0.0,
    1.0
)

policy_view_df["survival_need"] = np.clip(
    0.34 * policy_view_df["pred_water"] +
    0.33 * policy_view_df["pred_food"] +
    0.33 * policy_view_df["pred_shelter"],
    0.0,
    1.0
)

policy_view_df["medical_need"] = np.clip(
    0.55 * policy_view_df["pred_medical_help"] +
    0.25 * policy_view_df["pred_medical_products"] +
    0.20 * policy_view_df["pred_hospitals"],
    0.0,
    1.0
)

policy_view_df["rescue_support"] = np.clip(
    0.50 * policy_view_df["pred_search_and_rescue"] +
    0.30 * policy_view_df["pred_transport"] +
    0.20 * policy_view_df["pred_hospitals"],
    0.0,
    1.0
)

policy_view_df["request_support"] = np.clip(
    0.50 * policy_view_df["pred_other_aid"] +
    0.30 * policy_view_df["pred_transport"] +
    0.20 * policy_view_df["pred_aid_centers"],
    0.0,
    1.0
)

policy_view_df["infrastructure_support"] = np.clip(
    0.50 * policy_view_df["pred_infrastructure_related"] +
    0.30 * policy_view_df["pred_buildings"] +
    0.20 * policy_view_df["pred_electricity"],
    0.0,
    1.0
)

policy_view_df["concrete_need"] = policy_view_df[["survival_need", "medical_need"]].max(axis=1)
policy_view_df["directness"] = policy_view_df["genre"].apply(genre_to_directness)

display(
    policy_view_df[
        [
            "message", "genre", "general_distress", "survival_need",
            "medical_need", "rescue_support", "request_support",
            "infrastructure_support", "concrete_need", "directness"
        ]
    ].head(10)
)


,message,genre,general_distress,survival_need,medical_need,rescue_support,request_support,infrastructure_support,concrete_need,directness
293,I would like to leave this country and at my parents house in the USA. We are alone in Haiti. This is my number,direct,0.457240,0.251541,0.045572,0.143180,0.294074,0.407177,0.251541,1.0
17589,"Of the 78 stations, 71 were using the seismic, six the hydroacoustic and one the infrasound technologies.",news,0.127727,0.011045,0.079881,0.088028,0.132871,0.119768,0.079881,0.1
287,Message reads : Survivors are still sleeping in the streets. Please bring some advices for them for us please thank you.,direct,0.807381,0.519117,0.154796,0.143223,0.411760,0.218073,0.519117,1.0
11887,"RT reakingNews: Chile update: Witnesses say two separate aftershocks rocked Santiago within minutes of each other, according to Reuters",social,0.328631,0.025918,0.323408,0.181305,0.278430,0.182868,0.323408,0.6
328,"Hi, I have a big problem I need to call overseas and tell them I'm ok but I have no cell phone minutes, how can you help me in that respect",direct,0.308307,0.023117,0.014390,0.044037,0.280098,0.017696,0.023117,1.0
17509,We have provided $41 million so far out of that monies to support agencies providing relief on the ground and for humanitarian commodities.,news,0.848732,0.358911,0.484658,0.243592,0.496372,0.182379,0.484658,0.1
18132,"""This assumption was accepted by nuclear plant operators and was not challenged by regulators or by the government,"" he said.",news,0.102730,0.014342,0.019588,0.084891,0.123270,0.145738,0.019588,0.1
2659,are you really going to leave me with no help?,direct,0.456480,0.063453,0.047761,0.088890,0.345528,0.045631,0.063453,1.0
17581,"The island state off the east coast of Africa was described as the ""great red island"" by Marco Polo due to the abundant rust-red laterite soil.",news,0.374742,0.042662,0.265209,0.181177,0.284634,0.277134,0.265209,0.1
25991,"According to a UN Electoral Assistance Mission to Zimbabwe which ended last month, the voters roll is ""flawed"", with 10-25 percent of the names on the list deceased and an estimated 2 million out of 5.6 registered voters having moved constituencies since the last election in 1995 without proper registration.",news,0.373821,0.103406,0.236278,0.077520,0.169286,0.104316,0.236278,0.1


In [ ]:
# Fuzzy inference functions for the final policies

def infer_general_aid(row):
    gd = fuzzify_scalar(row["general_distress"])
    sn = fuzzify_scalar(row["survival_need"])
    mn = fuzzify_scalar(row["medical_need"])
    rs = fuzzify_scalar(row["request_support"])
    dr = fuzzify_scalar(row["directness"])

    rules = [
        (min(dr["very_high"], gd["very_high"], max(sn["high"], mn["high"])), "very_high"),
        (min(dr["high"], gd["high"], sn["very_high"]), "very_high"),
        (min(dr["high"], gd["very_high"], rs["high"]), "high"),
        (min(dr["high"], gd["high"], max(sn["high"], mn["medium"])), "high"),
        (min(max(dr["high"], dr["medium"]), gd["medium"], max(sn["medium"], mn["medium"])), "medium"),
        (min(dr["medium"], gd["high"], rs["medium"]), "medium"),
        (max(dr["low"], min(gd["low"], sn["low"], mn["low"])), "low"),
    ]
    return mamdani_infer(rules)

def infer_urgent_medical_response(row):
    gd = fuzzify_scalar(row["general_distress"])
    mn = fuzzify_scalar(row["medical_need"])
    rs = fuzzify_scalar(row["rescue_support"])
    sn = fuzzify_scalar(row["survival_need"])
    dr = fuzzify_scalar(row["directness"])

    rules = [
        (min(dr["very_high"], gd["high"], mn["very_high"], rs["high"]), "very_high"),
        (min(dr["high"], gd["very_high"], mn["high"], rs["high"]), "high"),
        (min(dr["high"], gd["high"], mn["very_high"], rs["medium"]), "high"),
        (min(dr["high"], gd["high"], mn["high"], rs["high"]), "high"),

        (min(dr["high"], gd["medium"], mn["high"], rs["medium"]), "medium"),
        (min(dr["very_high"], gd["high"], mn["high"], rs["low"]), "medium"),
        (min(dr["medium"], gd["high"], mn["very_high"], rs["high"]), "medium"),
        (min(dr["medium"], mn["very_high"], rs["very_high"]), "medium"),

        (max(
            dr["low"],
            mn["low"],
            rs["low"],
            min(sn["high"], mn["low"]),
            min(dr["medium"], mn["medium"], rs["low"]),
            min(dr["medium"], gd["low"])
        ), "low"),
    ]
    return mamdani_infer(rules)

def infer_shelter_supplies(row):
    gd = fuzzify_scalar(row["general_distress"])
    sn = fuzzify_scalar(row["survival_need"])
    rs = fuzzify_scalar(row["request_support"])
    infra = fuzzify_scalar(row["infrastructure_support"])
    dr = fuzzify_scalar(row["directness"])

    rules = [
        (min(dr["very_high"], sn["very_high"], max(rs["high"], gd["high"])), "very_high"),
        (min(dr["high"], sn["high"], rs["high"]), "high"),
        (min(dr["high"], sn["very_high"], gd["medium"]), "high"),
        (min(dr["high"], sn["high"], infra["medium"]), "medium"),
        (min(max(dr["high"], dr["medium"]), sn["medium"], rs["medium"]), "medium"),
        (min(dr["medium"], sn["high"], gd["medium"]), "medium"),
        (max(dr["low"], sn["low"], rs["low"]), "low"),
    ]
    return mamdani_infer(rules)

def infer_request_emulation(row):
    gd = fuzzify_scalar(row["general_distress"])
    cn = fuzzify_scalar(row["concrete_need"])
    rs = fuzzify_scalar(row["request_support"])
    dr = fuzzify_scalar(row["directness"])

    rules = [
        (min(dr["very_high"], max(gd["high"], gd["very_high"]), max(cn["high"], cn["very_high"])), "very_high"),
        (min(dr["high"], gd["high"], cn["very_high"]), "very_high"),
        (min(dr["high"], rs["very_high"], max(cn["medium"], cn["high"])), "very_high"),

        (min(dr["high"], gd["high"], cn["high"]), "high"),
        (min(dr["high"], gd["medium"], cn["high"]), "high"),
        (min(dr["high"], gd["high"], rs["high"]), "high"),
        (min(dr["high"], rs["high"], cn["medium"]), "high"),
        (min(dr["medium"], gd["high"], cn["high"]), "high"),
        (min(dr["medium"], gd["high"], rs["high"]), "high"),

        (min(max(dr["high"], dr["medium"]), gd["medium"], cn["medium"]), "medium"),
        (min(dr["medium"], gd["medium"], cn["high"]), "medium"),
        (min(dr["medium"], gd["high"], rs["medium"]), "medium"),
        (min(dr["high"], rs["medium"], cn["medium"]), "medium"),
        (min(dr["medium"], rs["high"], cn["medium"]), "medium"),

        (max(
            min(dr["low"], gd["low"]),
            min(dr["low"], cn["low"]),
            min(gd["low"], cn["low"], rs["low"])
        ), "low"),
    ]
    return mamdani_infer(rules)



In [ ]:
policy_view_df["policy_general_aid"] = policy_view_df.apply(infer_general_aid, axis=1)
policy_view_df["policy_urgent_medical_response"] = policy_view_df.apply(infer_urgent_medical_response, axis=1)
policy_view_df["policy_shelter_supplies"] = policy_view_df.apply(infer_shelter_supplies, axis=1)
policy_view_df["policy_request_emulation"] = policy_view_df.apply(infer_request_emulation, axis=1)

display(
    policy_view_df[
        [
            "message", "genre",
            "policy_general_aid",
            "policy_urgent_medical_response",
            "policy_shelter_supplies",
            "policy_request_emulation",
        ]
    ].head(10)
)


,message,genre,policy_general_aid,policy_urgent_medical_response,policy_shelter_supplies,policy_request_emulation
293,I would like to leave this country and at my parents house in the USA. We are alone in Haiti. This is my number,direct,0.000000,0.110098,0.136960,0.000000
17589,"Of the 78 stations, 71 were using the seismic, six the hydroacoustic and one the infrasound technologies.",news,0.110098,0.110098,0.110098,0.110098
287,Message reads : Survivors are still sleeping in the streets. Please bring some advices for them for us please thank you.,direct,0.000000,0.114119,0.000000,0.000000
11887,"RT reakingNews: Chile update: Witnesses say two separate aftershocks rocked Santiago within minutes of each other, according to Reuters",social,0.428567,0.121481,0.110098,0.428567
328,"Hi, I have a big problem I need to call overseas and tell them I'm ok but I have no cell phone minutes, how can you help me in that respect",direct,0.000000,0.110098,0.110098,0.000000
17509,We have provided $41 million so far out of that monies to support agencies providing relief on the ground and for humanitarian commodities.,news,0.110098,0.110098,0.110098,0.000000
18132,"""This assumption was accepted by nuclear plant operators and was not challenged by regulators or by the government,"" he said.",news,0.110098,0.110098,0.110098,0.110098
2659,are you really going to leave me with no help?,direct,0.000000,0.110098,0.110098,0.000000
17581,"The island state off the east coast of Africa was described as the ""great red island"" by Marco Polo due to the abundant rust-red laterite soil.",news,0.110098,0.110098,0.110098,0.140219
25991,"According to a UN Electoral Assistance Mission to Zimbabwe which ended last month, the voters roll is ""flawed"", with 10-25 percent of the names on the list deceased and an estimated 2 million out of 5.6 registered voters having moved constituencies since the last election in 1995 without proper registration.",news,0.110098,0.110098,0.110098,0.133426


In [ ]:
# Top 10 per fuzzy policy

pd.set_option("display.max_colwidth", None)

def top_policy_table(df, policy_col, n=10):
    return (
        df.sort_values(policy_col, ascending=False)
        .drop_duplicates(subset=["message_clean"])[
            ["message", "genre", policy_col]
        ]
        .head(n)
        .reset_index(drop=True)
    )

def display_full_messages(df):
    display(
        df.style.set_properties(
            subset=["message"],
            **{
                "white-space": "pre-wrap",
                "text-align": "left",
                "max-width": "900px",
            }
        )
    )

print("Top 10: policy_general_aid")
display_full_messages(top_policy_table(policy_view_df, "policy_general_aid", 10))

print("Top 10: policy_urgent_medical_response")
display_full_messages(top_policy_table(policy_view_df, "policy_urgent_medical_response", 10))

print("Top 10: policy_shelter_supplies")
display_full_messages(top_policy_table(policy_view_df, "policy_shelter_supplies", 10))


Top 10: policy_general_aid


,message,genre,policy_general_aid
0,"Help please, we need help at Christ-Roi street, Impasse Cedor. We need food, water, medical care and tents because there are two newborn babies in the. ..",direct,0.931223
1,"A.T.S.C.T: we need aid at Tabarre 48 (bwablan). We don't have tents, we don't have food. Please help us, help us find some of the aid they sent for the country.",direct,0.930568
2,"I would like to donate clothing , non-perishable food items and baby items , that we collect here in our community . Also , may be available to volunteer in a shelter during Thanksgiving weekend ( November 22 - 26 ) . Not physically big or strong but can help in other ways .",direct,0.930420
3,"I am collecting donations from the area and I can drop off the items this week including : non-perishables , baby supplies , Hygiene products , cleaning supplies . As of now I am planning on dropping off the items ( sorted ) to the shelters listed on the site . Let me know if there is a better location to drop these items off .",direct,0.928511
4,"Flood has devastated our village Kachipul much. We have lost our crop, house, job and every thing. But the government hasn't provided any help to us. Location : vilage - kachipul district:kambr shadadkot. Please visit our city kachipul.",direct,0.928333
5,"How we can help my camp? We need mobil clinical, because we have pregnant women with child who need care.",direct,0.928152
6,We need help at Place Mais-Gate. We have a sick baby.,direct,0.928016
7,"We can help clean streets , houses , prepare and deliver food . We have a push broom and rake . We can do anything needed that 's not heavy lifting",direct,0.927219
8,"I can help by gathering items needed ( clothes , food , jackets , etc . ) and delivering to locatons that need it .",direct,0.927132
9,"I need to Help me because my house broken. My food, tent, mattress",direct,0.926761


Top 10: policy_urgent_medical_response


,message,genre,policy_urgent_medical_response
0,"RT pacetesix15: #Chile needs fund relief, Imagine?! RT @wiredscience: Chile quake moved the city of Concepcion 10 feet to the west. ht ..",social,0.430000
1,Ppl being evacuated from NYU hospital .. Women who just gave birth too .. #SandyProblems,social,0.429469
2,I can get supplies from the pharmacy if needed ?,direct,0.334974
3,"I am a 5th grade teacher in California . My class would like to gather supplies , but we want to make sure they go to someplace in need . We can gather blankets , beanies , scarves , gloves , and hygiene products if anyone is interested . Flashlights and batteries , as well as school supplies are also a possibility . We can get it shipped ASAP as long as we have contact information . Let us know , we want to contribute !",direct,0.332004
4,"At 81 Mahotiere we need nourishment, water and health care.",direct,0.328680
5,The medicine should be distributed in all the departments ( counties ) because the hospital ( s ). ..,direct,0.325444
6,Here is a list of the items we have to offer : *Men 's & Women 's coats and winter wear *Children 's coats and winter wear ( boys & girls ) *Children 's and baby toys and stuffed animals ( all new w/tags ) *Blankets ( all new w/tags ) *Toiletries/Hygiene products *Water *Cleaning supplies,direct,0.325431
7,"To all institutions receiving this message and that need a health agent to work in temporary shelters and cleanup of the sanitary installations and water treatment, etc, don't forget ( me )",direct,0.323932
8,"This is only for cancer patients undergoing chemotherapy who have been displaced by the storm . We provide cancer patients undergoing chemo with kits of products to help manage side effects such as nausea , hair loss and oral care issues . We would love to get kits to people who have lost so much during an already difficult time .",direct,0.319763
9,"We are about 500 to 600 people in the temporary shelter of grand goave. and every day there are more and more people coming. We have a lot of issues such as drinking water, food, medicine,",direct,0.316537


Top 10: policy_shelter_supplies


,message,genre,policy_shelter_supplies
0,"Apeal Pir Ab Wahid village pir pir Dino u/c kenjar Taluka sujawal distt Thatta sir we are in need of clean drinking water,blanket,warm clothing,or croks iron bed",direct,0.929917
1,"we need packs of sanitaries,shelter food drink water",direct,0.929548
2,"I am place dessaline, near the champ de mars, we need water and food, all my family is in the street",direct,0.928844
3,we are in la plenn. we no food or water we are living off of god. we are waiting for help,direct,0.928018
4,"sir we are requesting to the UN people that if yo bring aid to sindh then please you people should distribute it. in our dist NGOs work on the source of MPA MNAK.We don't have tent, clean drinkable water and food.",direct,0.927662
5,Distribution of water . Helping in shelters and temporary sites,direct,0.927627
6,"Leogane is hardly hit, we need water and food",direct,0.927078
7,"My house crushed my child. im hungry and I dont have clean water to drink, i need so much. Im in Gresye between Billy beach. ..",direct,0.926980
8,food no water in the street we are we have people w/us that needs medical attention.,direct,0.926973
9,"Please ,. .. we are in Leogane, we have not got water or tent or food please. .. we ok, thanks in advace, send me an answer",direct,0.926841


In [ ]:
request_policy_rows = []

for threshold in np.arange(0.10, 0.91, 0.05):
    preds = (policy_view_df["policy_request_emulation"] >= threshold).astype(int)

    request_policy_rows.append({
        "threshold": round(float(threshold), 2),
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "positives_predicted": int(preds.sum()),
    })

request_policy_df = pd.DataFrame(request_policy_rows).sort_values("f1", ascending=False)
display(request_policy_df)

best_request_threshold = float(request_policy_df.iloc[0]["threshold"])
print("Best request-emulation threshold:", best_request_threshold)


,threshold,accuracy,precision,recall,f1,positives_predicted
8,0.50,0.857524,0.725610,0.265922,0.389207,328
7,0.45,0.856952,0.719033,0.265922,0.388254,331
9,0.55,0.857906,0.740385,0.258101,0.382767,312
15,0.85,0.857906,0.741935,0.256983,0.381743,310
14,0.80,0.857906,0.741935,0.256983,0.381743,310
12,0.70,0.857906,0.741935,0.256983,0.381743,310
11,0.65,0.857906,0.741935,0.256983,0.381743,310
10,0.60,0.857906,0.741935,0.256983,0.381743,310
13,0.75,0.857906,0.741935,0.256983,0.381743,310
16,0.90,0.857906,0.741935,0.256983,0.381743,310


Best request-emulation threshold: 0.5
